# 06. XGBoost Baseline

This notebook will be implemented in the next project stage.

In [1]:
from pathlib import Path
import sys

import pandas as pd

## Environment Setup

In [ ]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root configured.")

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
print("Processed directory configured.")

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EXPERIMENTS_DIR = ARTIFACTS_DIR / "experiments"

EXPERIMENT_ID = "xgboost_v0_raw_minimal_cv5"
RUN_TRAINING = False

Project root configured.
Processed directory configured.


In [3]:
from src.churn_ml.features import load_dataset
from src.churn_ml.modeling import (
    ExperimentConfig,
    load_experiment,
    save_experiment_result,
    build_experiment_summary,
)
from src.churn_ml.models.xgboost import (
    XGBoostConfig,
    run_xgboost_experiment,
)
from src.churn_ml.metrics import (
    evaluate_cross_fitted_thresholds,
    optimize_threshold_by_folds,
    summarize_threshold_plateau,
)

c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset Loading

In [4]:
dataset = load_dataset(
    version="v0_raw_minimal",
    processed_dir=PROCESSED_DIR,
)

print(f"Dataset version: {dataset.version}")
print(f"Train shape:     {dataset.X_train.shape}")
print(f"Target shape:    {dataset.y_train.shape}")
print(f"Test shape:      {dataset.X_test.shape}")

Dataset version: v0_raw_minimal
Train shape:     (10000, 205)
Target shape:    (10000,)
Test shape:      (2500, 205)


## XGBoost Baseline

In [5]:
xgboost_config = XGBoostConfig.default()
experiment_config = ExperimentConfig.default()

if RUN_TRAINING:
    xgboost_experiment = run_xgboost_experiment(
        dataset=dataset,
        config=experiment_config,
        model_config=xgboost_config,
        experiment_id=EXPERIMENT_ID,
    )

    save_experiment_result(
        result=xgboost_experiment.result,
        experiments_dir=EXPERIMENTS_DIR,
        fold_metrics=xgboost_experiment.fold_metrics,
        oof_predictions=xgboost_experiment.oof_predictions,
        test_predictions=xgboost_experiment.test_predictions,
        overwrite=True,
    )
else:
    xgboost_experiment = load_experiment(
        experiment_id=EXPERIMENT_ID,
        experiments_dir=EXPERIMENTS_DIR,
    )

Starting XGBoost cross-validation: 5 folds, 10,000 rows, 205 features


XGBoost CV: 100%|██████████| 5/5 [01:01<00:00, 12.22s/fold, balanced_accuracy=0.7744, best_iteration=1248, roc_auc=0.9569]


## Experiment Results

In [6]:
display(xgboost_experiment.fold_metrics)

display(
    pd.Series(
        xgboost_experiment.result.metrics,
        name="value",
    ).to_frame()
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,decision_threshold,training_time_seconds,best_iteration
0,1,0.782084,0.966656,0.834349,0.169058,0.5,14.861379,1443
1,2,0.771930,0.958346,0.814486,0.189615,0.5,11.341073,1056
2,3,0.795685,0.961886,0.821379,0.166895,0.5,7.781775,621
3,4,0.799131,0.960236,0.822506,0.177684,0.5,13.528517,1236
4,5,0.774418,0.956909,0.798295,0.195725,0.5,12.584407,1248


,value
balanced_accuracy,0.784650
roc_auc,0.960331
average_precision,0.816148
log_loss,0.179795
decision_threshold,0.500000
optimized_threshold_oof,0.041000
optimized_balanced_accuracy_oof,0.890959
balanced_accuracy_mean,0.784650
balanced_accuracy_std,0.012294
roc_auc_mean,0.960807


In [7]:
summary = build_experiment_summary(xgboost_experiment.result)

display(summary.style.format({"value": "{:.4f}"}))

,metric,value
0,Balanced Accuracy @ default threshold,0.7846
1,Optimized OOF Balanced Accuracy,0.8910
2,Optimized OOF threshold,0.0410
3,ROC-AUC,0.9603
4,Average Precision,0.8161
5,Log Loss,0.1798
6,"Training time, minutes",1.0189


In [8]:
fold_columns = [
    "fold",
    "balanced_accuracy",
    "roc_auc",
    "average_precision",
    "log_loss",
    "best_iteration",
    "training_time_seconds",
]

display(
    xgboost_experiment.fold_metrics[fold_columns].style.format(
        {
            "balanced_accuracy": "{:.4f}",
            "roc_auc": "{:.4f}",
            "average_precision": "{:.4f}",
            "log_loss": "{:.4f}",
            "training_time_seconds": "{:.1f}",
        }
    )
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,best_iteration,training_time_seconds
0,1,0.7821,0.9667,0.8343,0.1691,1443,14.9
1,2,0.7719,0.9583,0.8145,0.1896,1056,11.3
2,3,0.7957,0.9619,0.8214,0.1669,621,7.8
3,4,0.7991,0.9602,0.8225,0.1777,1236,13.5
4,5,0.7744,0.9569,0.7983,0.1957,1248,12.6


## Threshold Stability

In [9]:
fold_thresholds = optimize_threshold_by_folds(
    fold_metrics=xgboost_experiment.fold_metrics,
    oof_predictions=xgboost_experiment.oof_predictions,
)

display(
    fold_thresholds.style.format(
        {
            "optimized_threshold": "{:.3f}",
            "optimized_balanced_accuracy": "{:.4f}",
        }
    )
)

print(f"Threshold mean: {fold_thresholds['optimized_threshold'].mean():.3f}")

print(f"Threshold std:  {fold_thresholds['optimized_threshold'].std(ddof=1):.3f}")

,fold,optimized_threshold,optimized_balanced_accuracy
0,1,0.018,0.9060
1,2,0.053,0.8872
2,3,0.041,0.8936
3,4,0.043,0.8921
4,5,0.027,0.8930


Threshold mean: 0.036
Threshold std:  0.014


## Cross-Fitted Threshold Evaluation

In [10]:
cross_fitted_result = evaluate_cross_fitted_thresholds(
    xgboost_experiment.oof_predictions
)

cross_fitted_fold_metrics = cross_fitted_result.fold_metrics

display(
    cross_fitted_fold_metrics.style.format(
        {
            "threshold_cross_fitted": "{:.3f}",
            "balanced_accuracy_cross_fitted": "{:.4f}",
            "threshold_fold_optimal": "{:.3f}",
            "balanced_accuracy_fold_optimal": "{:.4f}",
            "balanced_accuracy_regret": "{:.4f}",
        }
    )
)

,fold,threshold_cross_fitted,balanced_accuracy_cross_fitted,threshold_fold_optimal,balanced_accuracy_fold_optimal,balanced_accuracy_regret
0,1,0.041,0.8987,0.018,0.9060,0.0073
1,2,0.040,0.8804,0.053,0.8872,0.0068
2,3,0.041,0.8936,0.041,0.8936,0.0000
3,4,0.040,0.8904,0.043,0.8921,0.0017
4,5,0.041,0.8905,0.027,0.8930,0.0025


In [11]:
optimized_oof_balanced_accuracy = xgboost_experiment.result.metrics[
    "optimized_balanced_accuracy_oof"
]

optimism_gap = optimized_oof_balanced_accuracy - cross_fitted_result.balanced_accuracy

print(
    f"Cross-fitted OOF Balanced Accuracy: {cross_fitted_result.balanced_accuracy:.4f}"
)
print(
    "Mean cross-fitted threshold:         "
    f"{cross_fitted_fold_metrics['threshold_cross_fitted'].mean():.3f}"
)
print(
    "Cross-fitted threshold std:          "
    f"{cross_fitted_fold_metrics['threshold_cross_fitted'].std(ddof=1):.3f}"
)
print(
    "Mean Balanced Accuracy regret:       "
    f"{cross_fitted_fold_metrics['balanced_accuracy_regret'].mean():.4f}"
)
print(f"OOF threshold-selection optimism:    {optimism_gap:.4f}")

Cross-fitted OOF Balanced Accuracy: 0.8907
Mean cross-fitted threshold:         0.041
Cross-fitted threshold std:          0.001
Mean Balanced Accuracy regret:       0.0037
OOF threshold-selection optimism:    0.0002


## Threshold Plateau

In [12]:
threshold_plateau = summarize_threshold_plateau(
    y_true=xgboost_experiment.oof_predictions["target"],
    probabilities=xgboost_experiment.oof_predictions["probability"].to_numpy(),
    tolerance=0.001,
)

threshold_plateau_summary = pd.DataFrame(
    [
        {
            "best_threshold": threshold_plateau.best_threshold,
            "best_balanced_accuracy": (threshold_plateau.best_balanced_accuracy),
            "lower_threshold": threshold_plateau.lower_threshold,
            "upper_threshold": threshold_plateau.upper_threshold,
            "midpoint_threshold": (threshold_plateau.midpoint_threshold),
            "plateau_width": threshold_plateau.width,
            "tolerance": threshold_plateau.tolerance,
        }
    ]
)

display(
    threshold_plateau_summary.style.format(
        {
            "best_threshold": "{:.3f}",
            "best_balanced_accuracy": "{:.4f}",
            "lower_threshold": "{:.3f}",
            "upper_threshold": "{:.3f}",
            "midpoint_threshold": "{:.3f}",
            "plateau_width": "{:.3f}",
            "tolerance": "{:.4f}",
        }
    )
)

,best_threshold,best_balanced_accuracy,lower_threshold,upper_threshold,midpoint_threshold,plateau_width,tolerance
0,0.041,0.8910,0.039,0.041,0.040,0.002,0.0010


In [13]:
fold_plateau_records = []

for fold_number in sorted(xgboost_experiment.oof_predictions["fold"].unique()):
    fold_data = xgboost_experiment.oof_predictions.loc[
        xgboost_experiment.oof_predictions["fold"] == fold_number
    ]

    fold_plateau = summarize_threshold_plateau(
        y_true=fold_data["target"],
        probabilities=fold_data["probability"].to_numpy(),
        tolerance=0.001,
    )

    fold_plateau_records.append(
        {
            "fold": int(fold_number),
            "best_threshold": fold_plateau.best_threshold,
            "lower_threshold": fold_plateau.lower_threshold,
            "upper_threshold": fold_plateau.upper_threshold,
            "midpoint_threshold": (fold_plateau.midpoint_threshold),
            "plateau_width": fold_plateau.width,
            "best_balanced_accuracy": (fold_plateau.best_balanced_accuracy),
        }
    )

fold_plateaus = pd.DataFrame(fold_plateau_records)

display(
    fold_plateaus.style.format(
        {
            "best_threshold": "{:.3f}",
            "lower_threshold": "{:.3f}",
            "upper_threshold": "{:.3f}",
            "midpoint_threshold": "{:.3f}",
            "plateau_width": "{:.3f}",
            "best_balanced_accuracy": "{:.4f}",
        }
    )
)

,fold,best_threshold,lower_threshold,upper_threshold,midpoint_threshold,plateau_width,best_balanced_accuracy
0,1,0.018,0.018,0.018,0.018,0.000,0.9060
1,2,0.053,0.051,0.059,0.055,0.008,0.8872
2,3,0.041,0.040,0.041,0.041,0.001,0.8936
3,4,0.043,0.020,0.053,0.037,0.033,0.8921
4,5,0.027,0.026,0.036,0.031,0.010,0.8930


In [14]:
selected_threshold = xgboost_experiment.result.metrics["optimized_threshold_oof"]

selected_threshold_inside_plateau = (
    threshold_plateau.lower_threshold
    <= selected_threshold
    <= threshold_plateau.upper_threshold
)

print(f"Selected OOF threshold:       {selected_threshold:.3f}")
print(
    "Near-optimal plateau:        "
    f"[{threshold_plateau.lower_threshold:.3f}, "
    f"{threshold_plateau.upper_threshold:.3f}]"
)
print(f"Selected threshold in plateau: {selected_threshold_inside_plateau}")

Selected OOF threshold:       0.041
Near-optimal plateau:        [0.039, 0.041]
Selected threshold in plateau: True


## Baseline Conclusion

In [15]:
final_threshold_summary = pd.DataFrame(
    [
        {
            "metric": "Balanced Accuracy @ 0.5",
            "value": xgboost_experiment.result.metrics["balanced_accuracy"],
        },
        {
            "metric": "Optimized OOF Balanced Accuracy",
            "value": xgboost_experiment.result.metrics[
                "optimized_balanced_accuracy_oof"
            ],
        },
        {
            "metric": "Cross-fitted Balanced Accuracy",
            "value": cross_fitted_result.balanced_accuracy,
        },
        {
            "metric": "Selected threshold",
            "value": selected_threshold,
        },
        {
            "metric": "Cross-fitted threshold mean",
            "value": cross_fitted_fold_metrics["threshold_cross_fitted"].mean(),
        },
        {
            "metric": "Threshold plateau lower bound",
            "value": threshold_plateau.lower_threshold,
        },
        {
            "metric": "Threshold plateau upper bound",
            "value": threshold_plateau.upper_threshold,
        },
        {
            "metric": "ROC-AUC",
            "value": xgboost_experiment.result.metrics["roc_auc"],
        },
        {
            "metric": "Average Precision",
            "value": xgboost_experiment.result.metrics["average_precision"],
        },
    ]
)

display(final_threshold_summary.style.format({"value": "{:.4f}"}))

,metric,value
0,Balanced Accuracy @ 0.5,0.7846
1,Optimized OOF Balanced Accuracy,0.8910
2,Cross-fitted Balanced Accuracy,0.8907
3,Selected threshold,0.0410
4,Cross-fitted threshold mean,0.0406
5,Threshold plateau lower bound,0.0390
6,Threshold plateau upper bound,0.0410
7,ROC-AUC,0.9603
8,Average Precision,0.8161


In [16]:
expected_test_predictions = (
    xgboost_experiment.test_predictions["probability"] >= selected_threshold
).astype(int)

assert expected_test_predictions.equals(
    xgboost_experiment.test_predictions["prediction_optimized_oof"]
)

test_positive_rate = expected_test_predictions.mean()

print(f"Selected threshold:       {selected_threshold:.3f}")
print(f"Test positive rate:       {test_positive_rate:.2%}")
print(f"Predicted positive rows:  {expected_test_predictions.sum():,}")
print(f"Total test rows:          {len(expected_test_predictions):,}")

Selected threshold:       0.041
Test positive rate:       22.60%
Predicted positive rows:  565
Total test rows:          2,500


In [17]:
experiment_dir = EXPERIMENTS_DIR / EXPERIMENT_ID
experiment_dir.mkdir(parents=True, exist_ok=True)

cross_fitted_fold_metrics.to_csv(
    experiment_dir / "cross_fitted_threshold_metrics.csv",
    index=False,
)

fold_plateaus.to_csv(
    experiment_dir / "threshold_plateaus_by_fold.csv",
    index=False,
)

final_threshold_summary.to_csv(
    experiment_dir / "threshold_summary.csv",
    index=False,
)

print(f"Threshold diagnostics saved to: {experiment_dir}")

Threshold diagnostics saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments\xgboost_v0_raw_minimal_cv5


## Feature Importance

Gain-based feature importance is calculated for each fitted XGBoost fold and then averaged across folds.

The importance values are normalized within each fold. Features that were not used in any tree split receive an importance of zero.

In [18]:
feature_importance_by_fold = []

for fold_number, model in enumerate(
    xgboost_experiment.fitted_models,
    start=1,
):
    importance_mapping = model.get_booster().get_score(importance_type="total_gain")

    fold_importance = pd.DataFrame(
        {
            "feature": dataset.X_train.columns,
            "importance": [
                float(importance_mapping.get(feature, 0.0))
                for feature in dataset.X_train.columns
            ],
            "fold": fold_number,
        }
    )

    total_importance = fold_importance["importance"].sum()

    if total_importance > 0:
        fold_importance["importance"] /= total_importance

    feature_importance_by_fold.append(fold_importance)

feature_importance_folds = pd.concat(
    feature_importance_by_fold,
    ignore_index=True,
)

feature_importance_summary = (
    feature_importance_folds.groupby(
        "feature",
        as_index=False,
    )
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std"),
        folds_used=("importance", lambda values: int((values > 0).sum())),
    )
    .sort_values(
        "importance_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    feature_importance_summary.head(30).style.format(
        {
            "importance_mean": "{:.4%}",
            "importance_std": "{:.4%}",
        }
    )
)

,feature,importance_mean,importance_std,folds_used
0,Var126,21.4209%,0.2758%,5
1,Var192,8.7330%,0.2419%,5
2,Var212,8.6653%,1.7220%,5
3,Var216,8.1000%,2.1118%,5
4,Var204,5.3452%,0.3284%,5
5,Var199,4.8540%,0.2531%,5
6,Var197,3.0277%,0.3755%,5
7,Var74,2.8890%,0.1585%,5
8,Var73,2.8779%,0.3441%,5
9,Var228,2.4791%,0.3132%,5


In [19]:
feature_importance_path = experiment_dir / "feature_importance.csv"

feature_importance_summary.to_csv(
    feature_importance_path,
    index=False,
)

print(f"Feature importance saved to: {feature_importance_path}")

Feature importance saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments\xgboost_v0_raw_minimal_cv5\feature_importance.csv
